In [2]:
# =====================================================
# LIFE EXPECTANCY ANALYSIS — ROBUST INTERNSHIP VERSION
# Handles WHO 2000–2015 dataset + similar datasets
# =====================================================

import os
import sys
import warnings
warnings.filterwarnings("ignore")

# ================= INSTALL PACKAGES ==================
def install(pkg):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

required = ["pandas","numpy","matplotlib","seaborn",
            "scikit-learn","statsmodels","xgboost","streamlit"]

for r in required:
    try:
        __import__(r)
    except:
        install(r)

# ================= IMPORTS ==================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from xgboost import XGBRegressor

RANDOM_STATE = 42

# =====================================================
# 1️⃣ LOAD DATA
# =====================================================
df = pd.read_csv("Life Expectancy Data.csv")

# Clean column names (very important)
df.columns = df.columns.str.strip()

print("Columns detected:")
print(df.columns.tolist())

# Detect target safely
if "Life expectancy" in df.columns:
    target = "Life expectancy"
else:
    target = [c for c in df.columns if "life" in c.lower()][0]

print("Target column:", target)

# Encode Status if exists
if "Status" in df.columns:
    df["Status"] = df["Status"].astype(str)
    df["Status_encoded"] = LabelEncoder().fit_transform(df["Status"])

# =====================================================
# 2️⃣ ADVANCED FEATURE ENGINEERING
# =====================================================

# Immunization index (if available)
immun_cols = [c for c in ["Hepatitis B","Polio","Diphtheria"] if c in df.columns]
if immun_cols:
    df["Immunization_Index"] = df[immun_cols].mean(axis=1)

# Log transforms
for col in ["GDP","Population"]:
    if col in df.columns:
        df[col+"_log"] = np.log1p(df[col])

# Lag features (panel-based)
if "Country" in df.columns:
    df = df.sort_values(["Country","Year"])
    for col in ["GDP","Total expenditure","Schooling"]:
        if col in df.columns:
            df[col+"_lag1"] = df.groupby("Country")[col].shift(1)

# Growth feature
if "GDP" in df.columns:
    df["GDP_growth"] = df.groupby("Country")["GDP"].pct_change()

# =====================================================
# 3️⃣ IMPUTATION (MICE)
# =====================================================
numeric_cols = df.select_dtypes(include=np.number).columns
imputer = IterativeImputer(random_state=RANDOM_STATE)
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

# =====================================================
# 4️⃣ MODEL DIAGNOSTICS (OLS + VIF + BP TEST)
# =====================================================
print("\n===== OLS MODEL + DIAGNOSTICS =====")

exclude_cols = ["Country","Year","Status"]
features = [c for c in df.columns if c not in exclude_cols + [target]]

X = df[features]
y = df[target]

X_const = sm.add_constant(X)
ols_model = sm.OLS(y, X_const).fit()
print(ols_model.summary())

# VIF
vif_df = pd.DataFrame()
vif_df["Feature"] = X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i)
                 for i in range(X.shape[1])]
print("\nTop Multicollinearity Risks:")
print(vif_df.sort_values("VIF", ascending=False).head())

# Breusch-Pagan test
bp_test = het_breuschpagan(ols_model.resid, X_const)
print("\nBreusch-Pagan p-value:", bp_test[1])

# =====================================================
# 5️⃣ TIME-AWARE CROSS VALIDATION
# =====================================================
print("\n===== TIME SERIES CV =====")

if "Year" in df.columns:
    df = df.sort_values("Year")

tscv = TimeSeriesSplit(n_splits=5)

rf = RandomForestRegressor(random_state=RANDOM_STATE)

param_grid = {
    "n_estimators":[200,300,400],
    "max_depth":[5,10,None],
    "min_samples_split":[2,5,10]
}

rf_search = RandomizedSearchCV(rf,
                               param_grid,
                               n_iter=10,
                               cv=tscv,
                               scoring="r2",
                               n_jobs=-1,
                               random_state=RANDOM_STATE)

rf_search.fit(X,y)

best_rf = rf_search.best_estimator_
print("Best RF CV R2:", rf_search.best_score_)

# XGBoost model
xgb = XGBRegressor(objective="reg:squarederror",
                   random_state=RANDOM_STATE)
xgb.fit(X,y)

# =====================================================
# 9️⃣ SEGMENTATION ANALYSIS
# =====================================================
print("\n===== SEGMENTATION (Developed vs Developing) =====")

if "Status" in df.columns:
    for group in df["Status"].unique():
        seg = df[df["Status"]==group]
        if len(seg) > 30:
            X_seg = seg[features]
            y_seg = seg[target]
            model = RandomForestRegressor(random_state=RANDOM_STATE)
            model.fit(X_seg,y_seg)
            print(group,"R2:",round(model.score(X_seg,y_seg),3))

# =====================================================
# 12️⃣ POLICY INSIGHTS
# =====================================================
print("\n===== POLICY INSIGHTS =====")

importances = pd.Series(best_rf.feature_importances_, index=X.columns)
top = importances.sort_values(ascending=False).head(7)

print("Top Drivers of Life Expectancy:")
print(top)

print("\nPolicy Recommendations:")
print("- Strengthen immunization programs in low-performing countries.")
print("- Improve schooling years — strong long-term predictor.")
print("- Sustained healthcare expenditure increases life expectancy.")
print("- GDP growth helps but diminishing returns possible.")
print("- Reduce adult mortality and HIV/AIDS burden aggressively.")

# =====================================================
# 11️⃣ SAVE STRUCTURED OUTPUT
# =====================================================
os.makedirs("outputs", exist_ok=True)
df.to_csv("outputs/cleaned_data.csv", index=False)
importances.to_csv("outputs/feature_importance.csv")

# =====================================================
# 🔟 CREATE STREAMLIT DASHBOARD
# =====================================================
dashboard_code = """
import streamlit as st
import pandas as pd
import numpy as np

st.title("Life Expectancy Dashboard")

df = pd.read_csv("outputs/cleaned_data.csv")

country = st.selectbox("Select Country", df["Country"].unique())
country_df = df[df["Country"]==country]

st.line_chart(country_df.set_index("Year")["Life expectancy"])

st.subheader("Policy Simulator")

gdp = st.slider("GDP", float(df["GDP"].min()), float(df["GDP"].max()), float(df["GDP"].mean()))
school = st.slider("Schooling", float(df["Schooling"].min()), float(df["Schooling"].max()), float(df["Schooling"].mean()))
imm = st.slider("Immunization Index", float(df["Immunization_Index"].min()), float(df["Immunization_Index"].max()), float(df["Immunization_Index"].mean()))

prediction = 40 + 0.00002*gdp + 0.8*school + 0.1*imm
st.success(f"Predicted Life Expectancy: {round(prediction,2)} years")
"""

with open("dashboard.py","w") as f:
    f.write(dashboard_code)

print("\nDashboard created: dashboard.py")
print("Run with: streamlit run dashboard.py")

print("\n===== PROJECT COMPLETE — ROBUST VERSION =====")


Columns detected:
['Country', 'Year', 'Status', 'Life expectancy', 'Adult Mortality', 'infant deaths', 'Alcohol', 'percentage expenditure', 'Hepatitis B', 'Measles', 'BMI', 'under-five deaths', 'Polio', 'Total expenditure', 'Diphtheria', 'HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years', 'thinness 5-9 years', 'Income composition of resources', 'Schooling']
Target column: Life expectancy

===== OLS MODEL + DIAGNOSTICS =====
                            OLS Regression Results                            
Dep. Variable:        Life expectancy   R-squared:                       0.839
Model:                            OLS   Adj. R-squared:                  0.838
Method:                 Least Squares   F-statistic:                     608.9
Date:                Fri, 20 Feb 2026   Prob (F-statistic):               0.00
Time:                        07:05:02   Log-Likelihood:                -8100.0
No. Observations:                2938   AIC:                         1.625e+04
Df Residuals: 